## Week6Day5: Ghana T‑Bill Yield Curve Forecasting

### Summary
- **Goal:** Fine‑tune a chat model to predict 91‑, 182‑, and 364‑day Treasury Bill yields for Ghana.
- **Data:** `worldboss/bank-of-ghana-treasury-bills` (Hugging Face) — cleaned, pivoted to multi‑tenor yield curves.
- **Pipeline:** prepare_yield_curve → create_multi_tenor_training_data → export JSONL → upload & fine‑tune on OpenAI → evaluate predictor.
- **Deliverables:** training/validation JSONL files, fine‑tuned model (`gh_tbill_forecasting`), predictor function, evaluation metrics (MAE/RMSE).

In [ ]:
# imports
import os
import pandas as pd
import json
from dotenv import load_dotenv
from huggingface_hub import login
from openai import OpenAI
from datasets import load_dataset
from evaluator import Tester, evaluate

In [ ]:
# environment
load_dotenv(override=True)
hf_token = os.getenv("HF_TOKEN")
login(hf_token, add_to_git_credential=True)
openai = OpenAI()

In [ ]:
# Load the dataset
dataset = load_dataset("worldboss/bank-of-ghana-treasury-bills", split="train")
df = dataset.to_pandas()

In [ ]:
# 2. Cleaning
df = df.dropna()
# Remove FXR NOTE and FXR BOND securities
df = df[~df["Security_Type"].isin(["FXR NOTE", "FXR BOND"])]

df = df.reset_index(drop=True)

print(df.head())

In [ ]:
def prepare_yield_curve(df):
    """
    Convert Bank of Ghana treasury dataset into
    yield curve format suitable for forecasting.

    Input columns:
    Issue_Date, Tender, Security_Type, Discount_Rate, Interest_Rate
    """

    # Convert date
    df["Date"] = pd.to_datetime(df["Issue_Date"], format="%d-%b-%y")

    # Normalize security type text
    df["Security_Type"] = df["Security_Type"].str.strip().str.upper()

    # Pivot dataset
    pivot_df = df.pivot_table(
        index="Date", columns="Security_Type", values="Interest_Rate", aggfunc="mean"
    ).reset_index()

    # Rename columns
    pivot_df = pivot_df.rename(
        columns={
            "91 DAY BILL": "91-Day Rate",
            "182 DAY BILL": "182-Day Rate",
            "364 DAY BILL": "364-Day Rate",
        }
    )

    # Keep only rows with all tenors
    pivot_df = pivot_df.dropna(subset=["91-Day Rate", "182-Day Rate", "364-Day Rate"])

    # Sort chronologically
    pivot_df = pivot_df.sort_values("Date").reset_index(drop=True)

    return pivot_df


In [ ]:
def create_multi_tenor_training_data(df, window_size=4):
    """
    Create JSONL training samples for treasury yield forecasting
    """

    jsonl_data = []

    for i in range(len(df) - window_size):

        history = df.iloc[i : i + window_size]
        target = df.iloc[i + window_size]

        history_lines = []

        for _, row in history.iterrows():
            history_lines.append(
                f"{row['Date'].date()}: "
                f"[91d: {row['91-Day Rate']:.2f}%, "
                f"182d: {row['182-Day Rate']:.2f}%, "
                f"364d: {row['364-Day Rate']:.2f}%]"
            )

        history_snapshot = "\n".join(history_lines)

        entry = {
            "messages": [
                {
                    "role": "system",
                    "content": "You are a specialized Ghanaian debt market analyst forecasting the Treasury Bill yield curve.",
                },
                {
                    "role": "user",
                    "content": (
                        f"Historical Yields:\n"
                        f"{history_snapshot}\n"
                        f"Predict rates for {target['Date'].date()}."
                    ),
                },
                {
                    "role": "assistant",
                    "content": (
                        f"91d: {target['91-Day Rate']:.2f}%, "
                        f"182d: {target['182-Day Rate']:.2f}%, "
                        f"364d: {target['364-Day Rate']:.2f}%"
                    ),
                },
            ]
        }

        jsonl_data.append(entry)

    return jsonl_data

In [ ]:
yield_curve = prepare_yield_curve(df)
training_data = create_multi_tenor_training_data(yield_curve, window_size=6)
print(training_data[0])


In [ ]:
def export_jsonl(data, file_name="ghana_tbill_training.jsonl"):
    """
    Save training samples to JSONL
    """
    with open(file_name, "w") as f:
        for row in data:
            f.write(json.dumps(row) + "\n")

In [ ]:
validation_data = training_data[:50]

In [ ]:
# write the training and validation data to a jsonl file
export_jsonl(training_data)
export_jsonl(validation_data, "ghana_tbill_validation.jsonl")


### OpenAI for fine-tunning

In [ ]:
def create_openai_file(file_name):
    """
    Create a file in OpenAI for fine-tuning
    """
    with open(file_name, "rb") as f:
        train_file = openai.files.create(file=f, purpose="fine-tune")
        return train_file

In [ ]:
# create the training and validation files
train_file = create_openai_file("ghana_tbill_training.jsonl")
validation_file = create_openai_file("ghana_tbill_validation.jsonl")

In [ ]:
print(train_file.id)
print(validation_file.id)

In [ ]:
openai.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=validation_file.id,
    model="gpt-4o-mini-2024-07-18",
    seed=42,
    hyperparameters={"n_epochs": 1, "batch_size": 1},
    suffix="gh_tbill_forecasting",
)

In [ ]:
# List most recent fine-tuning job to check status
job = openai.fine_tuning.jobs.list(limit=1).data[0]
print(f"Status: {job.id}")
print(f"Status: {job.status}")

In [ ]:
# Get detailed information about the job
openai.fine_tuning.jobs.retrieve(job.id)

In [ ]:
# View training events log (last 10 events)
openai.fine_tuning.jobs.list_events(fine_tuning_job_id=job.id, limit=10).data

In [ ]:
# Get the fine-tuned model ID (only works after training succeeds)
fine_tuned_model_name = openai.fine_tuning.jobs.retrieve(job.id).fine_tuned_model
fine_tuned_model_name

In [ ]:
def test_trained_model(prompt):
    """Create messages for treasury bill forecasting prediction - NO answer (model must predict)"""
    system_message = """
    You are a specialized Ghanaian debt market analyst forecasting the Treasury Bill yield curve.
    Respond with the predicted yield curve for the given date.
    The yield curve should be in the following format where answer is the predicted yield:
    [91d: answer%, 182d: answer%, 364d: answer%]
    """
    return [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt},
    ]

In [ ]:
def gpt_fine_tuned(training_data):
    """Classify banking query using fine-tuned model"""
    response = openai.chat.completions.create(
        model=fine_tuned_model_name,
        messages=training_data,
        seed=42,
    )
    intent = response.choices[0].message.content.strip()
    return intent

In [ ]:
print(gpt_fine_tuned(training_data[100]['messages']))

In [ ]:
def my_predictor(datapoint):
    # datapoint is a JSONL entry with "messages"
    messages = (
        datapoint["messages"]
        if isinstance(datapoint, dict) and "messages" in datapoint
        else datapoint
    )
    return gpt_fine_tuned(
        messages
    )  # returns something like "91d: 14.73%, 182d: 15.16%, 364d: 18.10%"

In [ ]:
evaluate(my_predictor, validation_data, size=50)